In [1]:
import numpy as np

from model import TronBatchModel
from controller import RandomController, GreedySpaceController

import trainer

In [2]:
# Environment parameters
WIDTH = 64
HEIGHT = 48


## 1. Smoke test the vectorized environment

In [3]:
env = TronBatchModel(width=WIDTH, height=HEIGHT, players=2, envs=1024, keep_owner=False, seed=0)
ctrl = RandomController(seed=0)

obs = env.observe_lite()
actions = ctrl.actions(env)
result = env.step(actions)

print("obs shape:", obs.shape)
print("actions shape:", actions.shape)
print("reward shape:", result.reward.shape)
print("done count:", result.done.sum())

obs shape: (1024, 2, 13)
actions shape: (1024, 2)
reward shape: (1024, 2)
done count: 0


## 2. Fast rollout helper

trainer.evaluate_controller evaluates a controller over many parallel games.
Use this for baselines, genetic algorithms, and quick policy comparisons.

In [4]:

print("Random:", trainer.evaluate_controller(RandomController(1), envs=2048))
print("Greedy:", trainer.evaluate_controller(GreedySpaceController(), envs=2048))

Random: {'mean_reward_per_player': array([-0.04931641, -0.41748047], dtype=float32), 'win_rate_per_player': array([0.9995117, 0.9995117], dtype=float32), 'mean_length': np.float64(499.50439453125)}
Greedy: {'mean_reward_per_player': array([-0.60058594, -0.6308594 ], dtype=float32), 'win_rate_per_player': array([0.51904297, 0.51708984], dtype=float32), 'mean_length': np.float64(510.1474609375)}


## Params

Prefer currently
hidden: 8
popsize and elite_count: 64-8

In [5]:
greedy_opponent = GreedySpaceController()

genetic_algo_params = {
    "hidden": 8,
    'generations': 10,
    'pop_size': 64,
    'elite_count': 8,
    'eval_envs': 1024,

    "players":2,
    "width": WIDTH,
    "height": HEIGHT,
    'seed': 42,

    'opponent': greedy_opponent,
}
ga_params = {
    **genetic_algo_params,
}

In [6]:
probe_env = TronBatchModel(
    width=genetic_algo_params["width"],
    height=genetic_algo_params["height"],
    players=genetic_algo_params["players"],
    envs=1)
obs_dim = probe_env.observe_lite().shape[-1]

policy = trainer.MLPPolicy(obs_dim, hidden=genetic_algo_params["hidden"], rng=0)
print("obs_dim:", obs_dim)
print("genome parameters:", policy.n_params)

obs_dim: 13
genome parameters: 139
{'mean_reward_per_player': array([-15.15918  ,  -1.2744141], dtype=float32), 'win_rate_per_player': array([0.99316406, 1.        ], dtype=float32), 'mean_length': np.float64(487.4775390625)}


## 4. Genetic algorithm starter

This is intentionally simple:

1. Keep a population of MLP genomes.
2. Evaluate each genome.
3. Keep elites.
4. Refill the population with mutation + crossover.

In [7]:
best_fitness, best_genome, obs_dim = trainer.run_ga(
    **genetic_algo_params
)

#0.7381
print("best fitness:", best_fitness)

gen=000 best=+0.6329 mean=+0.5674
gen=001 best=+0.6323 mean=+0.5705
gen=002 best=+0.6439 mean=+0.5836
gen=003 best=+0.6472 mean=+0.5832
gen=004 best=+0.6503 mean=+0.5786
gen=005 best=+0.6642 mean=+0.6082
gen=006 best=+0.6693 mean=+0.6109
gen=007 best=+0.6719 mean=+0.6146
gen=008 best=+0.6822 mean=+0.6462
gen=009 best=+0.6798 mean=+0.6492
best fitness: 0.68215234375


## 5. Save and reload a trained genome

This saves the best genome into `./tron_genomes/` using a filename that includes key parameters and the current date/time. It also writes a `.json` metadata sidecar with the training settings.


In [9]:
import numpy as np

temp_params = {
    "hidden": 8,
    'generations': 10,
    'pop_size': 64,
    'elite_count': 8,
    'eval_envs': 1024,

    "players":2,
    "width": WIDTH,
    "height": HEIGHT,
    'seed': 42,
}
genome_path, metadata_path = trainer.export_genome(
    best_genome,
    obs_dim=obs_dim,
    fitness=best_fitness,
    **temp_params,
)

loaded = np.load(genome_path)
trained_policy = trainer.MLPPolicy(obs_dim, hidden=ga_params["hidden"], genome=loaded)

print(trainer.evaluate_controller(trained_policy, envs=ga_params["eval_envs"]))


Saved genome:  tron_genomes\tron_20260519_154310.npy
Saved metadata: tron_genomes\tron_20260519_154310.json
{'mean_reward_per_player': array([ 128., -128.], dtype=float32), 'win_rate_per_player': array([1., 0.], dtype=float32), 'mean_length': np.float64(384.0)}


## 6. Watch the trained genome in the GUI

The current `play_live()` function creates its own controller, so for visualizing a custom trained policy,
use a tiny local loop like this. Run it only on a machine with tkinter GUI support.

In [13]:
from view import GameView
def watch_mixed_policy(policies, players=2, width=32, height=32, scale=16, fps=20, seed=0):
    board_model = TronBatchModel(width=width, height=height, players=players, envs=1, keep_owner=True, randomize_spawns=True, seed=seed)
    view = GameView(board_model, scale=scale, fps=fps)

    try:
        while view.poll():
            if view.take_restart_request():
                board_model.reset()
            if not board_model.done[0]:
                acts = np.zeros((1, board_model.players), dtype=np.int8)
                for i in range(len(policies)):
                    policy = policies[i]

                    policy_acts = policy.actions(board_model)
                    acts[:, i] = policy_acts[:, i]
                board_model.step(acts)
            view.render(board_model)
    finally:
        view.close()

In [15]:
watch_mixed_policy([trained_policy, greedy_opponent], width=ga_params["width"], height=ga_params["height"])